In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ VG360 · training notebook (CPU, single cell)                                  ║
# ║                                                                              ║
# ║  Training only — everything that runs on site is Rust (github.com/dhaneshkk/ ║
# ║  test_clifford). This cell produces the PPE head classifier the Rust runtime ║
# ║  loads, and the evidence that it is the model it claims to be:               ║
# ║                                                                              ║
# ║   data (your crops, or synthetic smoke test) → train / val / test split      ║
# ║   → Clifford-conv Cl(2,0) net + plain-CNN baseline, both trained with SAM    ║
# ║   → model selection on val MCC → test MCC with bootstrap 95 % CI, sharpness  ║
# ║   → fold Clifford conv → plain conv → ONNX; torch ≡ folded ≡ onnxruntime     ║
# ║   → model card (SHA-256, MCC, gate) + golden vectors for the Rust tests      ║
# ║   → Lean: the fold, MCC and SAM facts used here are theorems in lean/VG360   ║
# ║   → optional: ask the running console via OpenAI /v1/chat/completions       ║
# ║   → optional: audit real site footage (detector + PPE), harvest crops        ║
# ║                                                                              ║
# ║  VG360_FAST=1 → tiny smoke run (CI). In Colab the repo is cloned so outputs  ║
# ║  land in models/ and testdata/golden/ ready to commit.                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# ─────────────────────────────── §0 setup & config ───────────────────────────────
import sys, os, json, time, math, random, hashlib, subprocess, importlib.util, datetime, shutil
from pathlib import Path
from dataclasses import dataclass, asdict

def _pip(**pkgs):                                     # import name → pip name
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    if missing: subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
_pip(onnx="onnx", onnxruntime="onnxruntime", sklearn="scikit-learn", cv2="opencv-python-headless")
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F, onnx, onnxruntime as ort
from sklearn.metrics import matthews_corrcoef, confusion_matrix

IN_COLAB = "google.colab" in sys.modules
FAST = os.environ.get("VG360_FAST") == "1"

@dataclass
class Config:
    repo_url: str = "https://github.com/dhaneshkk/test_clifford"
    clone_repo: bool = IN_COLAB          # Colab: clone the repo so outputs go to models/ and testdata/golden/
    write_repo: bool = not FAST          # write the model, card and golden vectors into the repo checkout
    ppe_dir: str = os.environ.get("VG360_PPE_DIR", "")   # ImageFolder: helmet/ no_helmet/ (real site crops)
    voc_dir: str = ""                    # Pascal-VOC head boxes (e.g. SHWD 'hat'/'person') → converted into ppe_dir
    n_synth: int = 600 if FAST else 5000 # synthetic smoke-test set size (used only without real crops)
    size: int = 48
    epochs: int = 2 if FAST else 14
    batch: int = 128
    lr: float = 0.05
    sam_rho: float = 0.05                # SAM radius (Foret et al. 2021)
    label_smoothing: float = 0.05
    baseline: bool = True                # also train a plain CNN with matched conv parameters (SAM too)
    bootstrap: int = 200 if FAST else 2000
    seed: int = 7
    severity: str = "critical"           # PPE rule severity → MCC floor (vg360_rules::tier)
    run_lean: bool = False               # True: elan + Mathlib cache, `lake build` of lean/ (several minutes)
    console_url: str = os.environ.get("VG360_URL", "")   # e.g. http://127.0.0.1:3000 → §8 asks the console
    footage: str = os.environ.get("VG360_FOOTAGE", "")   # site video(s), comma-separated → §9 detector + PPE audit
    footage_crop: str = ""               # ffmpeg crop for screen recordings, e.g. "1333:750:3:48" (drops VMS window chrome)
    footage_fps: float = 1.0             # frames per second sampled from the footage
    detector_url: str = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.onnx"

CFG = Config()
FLOORS = {"critical": 0.85, "high": 0.75, "medium": 0.60, "low": 0.60}   # same map as vg360_rules::tier
random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
torch.set_num_threads(max(1, os.cpu_count() or 1))

def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "lean/VG360/Clifford.lean").exists() and (p / "crates").exists(): return p
    return None
ROOT = find_root()
if ROOT is None and CFG.clone_repo:
    subprocess.run(["git", "clone", "-q", "--depth", "1", CFG.repo_url, "/content/test_clifford"], check=True)
    ROOT = Path("/content/test_clifford")
OUT = ROOT if (ROOT is not None and CFG.write_repo) else Path(os.environ.get("VG360_OUT", Path.cwd() / "vg360_train_out"))
(OUT / "models").mkdir(parents=True, exist_ok=True); (OUT / "testdata/golden").mkdir(parents=True, exist_ok=True)
REPORT = {"config": asdict(CFG), "fast": FAST}
print(f"[§0] torch {torch.__version__} · onnx {onnx.__version__} · ort {ort.__version__} · repo {ROOT} · out {OUT}"
      + (" · FAST smoke run" if FAST else ""))

# ─────────────────────────────── §1 Lean: the table and the facts ────────────────
# lean/VG360/Clifford.lean proves this exact table is the Cl(2,0) geometric product (associative,
# e1²=e2²=1, e1e2=−e2e1, e12²=−1); lean/VG360/Train.lean proves folding = the trained layer
# (fold_correct, fold_conv), MCC ∈ [−1,1] / perfect = 1 / class-swap invariant, and that SAM's
# ε = ρ g/‖g‖ is the worst case of the linearized loss on the ρ-ball (sam_eps_maximizes).
GP_IDX = [[0, 1, 2, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 2, 1, 0]]          # idxTable_eq
GP_SGN = [[1, 1, 1, 1], [1, 1, 1, 1], [1, -1, 1, -1], [1, -1, 1, -1]]      # sgnTable_eq

def cl2_table():
    """bitmask blades 1,e1,e2,e12 with e_i² = +1: sign = reordering parity (independent of the Lean table)."""
    idx, sgn = [[0] * 4 for _ in range(4)], [[0] * 4 for _ in range(4)]
    for a in range(4):
        for b in range(4):
            s, x = 0, a >> 1
            while x: s += bin(x & b).count("1"); x >>= 1
            idx[a][b], sgn[a][b] = a ^ b, -1 if s & 1 else 1
    return idx, sgn
assert cl2_table() == (GP_IDX, GP_SGN), "Clifford table drifted from lean/VG360/Clifford.lean"
if ROOT is not None:
    lean_src = (ROOT / "lean/VG360/Clifford.lean").read_text()
    assert "[[0, 1, 2, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 2, 1, 0]]" in lean_src
    assert "[[1, 1, 1, 1], [1, 1, 1, 1], [1, -1, 1, -1], [1, -1, 1, -1]]" in lean_src

def check_lean():
    """elan + Mathlib cache, then `lake build` of the repo's lean/ project (what CI runs)."""
    elan = Path.home() / ".elan/bin"; env = dict(os.environ, PATH=f"{elan}:{os.environ['PATH']}")
    if not (elan / "lake").exists():
        subprocess.run("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -o /tmp/elan-init.sh"
                       " && sh /tmp/elan-init.sh -y --default-toolchain none", shell=True, check=True)
    lean = ROOT / "lean"
    subprocess.run("lake exe cache get", shell=True, check=True, cwd=lean, env=env)
    r = subprocess.run("lake build", shell=True, cwd=lean, env=env, capture_output=True, text=True)
    print("[§1] Lean:", "all theorems checked ✓" if r.returncode == 0 else "FAILED\n" + r.stdout[-3000:] + r.stderr[-3000:])
    return r.returncode == 0
REPORT["lean"] = check_lean() if (CFG.run_lean and ROOT is not None) else "checked in CI (lean job); CFG.run_lean=True to run here"
print(f"[§1] Cl(2,0) table = Lean table ✓ · Lean: {REPORT['lean']}")

# ─────────────────────────────── §2 data ─────────────────────────────────────────
def synth_heads(n, size, rng):
    """Synthetic gray head crops — a pipeline smoke test, NOT a substitute for real site data."""
    X = np.zeros((n, size, size), np.float32); y = rng.integers(0, 2, n)
    for i in range(n):
        img = np.full((size, size), rng.uniform(40, 200), np.float32)
        img += cv2.GaussianBlur(rng.normal(0, 25, (size, size)).astype(np.float32), (0, 0), 3)
        cx, cy = size / 2 + rng.normal(0, 2), size * 0.58 + rng.normal(0, 2); r = size * rng.uniform(.2, .27)
        skin = rng.uniform(70, 190)
        cv2.ellipse(img, (int(cx), int(cy)), (int(r), int(r * 1.25)), 0, 0, 360, skin, -1)
        if y[i]:   # helmet: wider smooth dome + brim, bright or dark shell, specular highlight
            shell = rng.choice([rng.uniform(190, 250), rng.uniform(25, 70)])
            cv2.ellipse(img, (int(cx), int(cy - r * .35)), (int(r * 1.22), int(r * 1.05)), 0, 180, 360, shell, -1)
            cv2.line(img, (int(cx - r * 1.45), int(cy - r * .3)), (int(cx + r * 1.45), int(cy - r * .3)), shell * .9, 2)
            cv2.circle(img, (int(cx - r * .4), int(cy - r * .9)), max(1, int(r * .18)), min(255, shell + 40), -1)
        else:      # hard negatives: dark hair cap / beanie (no brim, tighter fit)
            if rng.random() < .7:
                cv2.ellipse(img, (int(cx), int(cy - r * .3)), (int(r * 1.02), int(r * .95)), 0, 180, 360,
                            rng.uniform(20, 120), -1)
        M = cv2.getRotationMatrix2D((size / 2, size / 2), rng.uniform(-25, 25), rng.uniform(.85, 1.15))
        img = cv2.warpAffine(img, M, (size, size), borderMode=cv2.BORDER_REFLECT)
        img = cv2.GaussianBlur(img, (0, 0), rng.uniform(.3, 1.4)) + rng.normal(0, rng.uniform(3, 14), img.shape)
        X[i] = np.clip(img, 0, 255) / 255.0
    return X[:, None], y

def voc_to_crops(voc_root, out_dir, label_map={"hat": "helmet", "person": "no_helmet"}, size=48):
    """Pascal-VOC head boxes (e.g. SHWD: 'hat' / 'person') → ImageFolder crops for CFG.ppe_dir."""
    import xml.etree.ElementTree as ET
    voc_root, out_dir = Path(voc_root), Path(out_dir)
    for xml in (voc_root / "Annotations").glob("*.xml"):
        root = ET.parse(xml).getroot(); img = cv2.imread(str(voc_root / "JPEGImages" / root.findtext("filename")), 0)
        if img is None: continue
        for k, ob in enumerate(root.iter("object")):
            lab = label_map.get(ob.findtext("name"))
            if not lab: continue
            bb = [int(float(ob.find("bndbox").findtext(t))) for t in ("xmin", "ymin", "xmax", "ymax")]
            crop = img[max(bb[1], 0):bb[3], max(bb[0], 0):bb[2]]
            if crop.size < 64: continue
            (out_dir / lab).mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(out_dir / lab / f"{xml.stem}_{k}.png"), cv2.resize(crop, (size, size)))

def load_data(rng):
    if CFG.voc_dir and not CFG.ppe_dir:
        CFG.ppe_dir = str(OUT / "ppe_crops"); voc_to_crops(CFG.voc_dir, CFG.ppe_dir, size=CFG.size)
    d = Path(CFG.ppe_dir) if CFG.ppe_dir else None
    if d and (d / "helmet").exists() and (d / "no_helmet").exists():
        X, y = [], []
        for lab, sub in ((1, "helmet"), (0, "no_helmet")):
            for p in sorted((d / sub).glob("*")):
                im = cv2.imread(str(p), 0)
                if im is not None: X.append(cv2.resize(im, (CFG.size, CFG.size)).astype(np.float32) / 255); y.append(lab)
        return np.array(X)[:, None], np.array(y), "real:" + str(d)
    X, y = synth_heads(CFG.n_synth, CFG.size, rng); return X, y, "synthetic (smoke test)"

def stratified_split(y, rng, frac=(0.6, 0.2, 0.2)):
    parts = [[], [], []]
    for c in np.unique(y):
        idx = rng.permutation(np.flatnonzero(y == c)); a = int(len(idx) * frac[0]); b = a + int(len(idx) * frac[1])
        for k, s in enumerate((idx[:a], idx[a:b], idx[b:])): parts[k] += s.tolist()
    return [np.array(sorted(p)) for p in parts]

rng = np.random.default_rng(CFG.seed)
X, Y, SOURCE = load_data(rng)
TR, VA, TE = stratified_split(Y, rng)
REPORT["data"] = {"source": SOURCE, "n": int(len(Y)), "n_train": int(len(TR)), "n_val": int(len(VA)), "n_test": int(len(TE)),
                  "helmet_fraction": float(Y.mean())}
print(f"[§2] {SOURCE}: {len(Y)} crops → train {len(TR)} · val {len(VA)} · test {len(TE)} (helmet {Y.mean():.0%})")

# ─────────────────────────────── §3 models ───────────────────────────────────────
def cayley():
    """M[i, j, k] = sign if b_i b_j = ±b_k — the tensor fold_correct is stated over."""
    M = torch.zeros(4, 4, 4)
    for i in range(4):
        for j in range(4): M[i, j, GP_IDX[i][j]] = float(GP_SGN[i][j])
    return M

class CliffordConv2d(nn.Module):
    """Cl(2,0) multivector channels [1,e1,e2,e12]: out = Σ W ⊙geo x. Weight (Cout,Cin,4,k,k) folds into a real conv
    (4Cout,4Cin,k,k) — same FLOPs at inference, 4× fewer parameters (Lean: Train.fold_correct / fold_conv)."""
    def __init__(self, cin, cout, k=3, stride=1, padding=1):
        super().__init__()
        self.cin, self.cout, self.k, self.stride, self.padding = cin, cout, k, stride, padding
        self.weight = nn.Parameter(torch.randn(cout, cin, 4, k, k) * math.sqrt(2.0 / (cin * 4 * k * k)))
        self.bias = nn.Parameter(torch.zeros(cout))          # scalar part only
        self.register_buffer("M", cayley())
    def real_weight(self):
        return torch.einsum("ocixy,ijk->okcjxy", self.weight, self.M).reshape(self.cout * 4, self.cin * 4, self.k, self.k)
    def real_bias(self):
        b = torch.zeros(self.cout, 4, device=self.bias.device); b[:, 0] = self.bias; return b.reshape(-1)
    def forward(self, x): return F.conv2d(x, self.real_weight(), self.real_bias(), self.stride, self.padding)
    def folded(self):
        c = nn.Conv2d(self.cin * 4, self.cout * 4, self.k, self.stride, self.padding)
        with torch.no_grad(): c.weight.copy_(self.real_weight()); c.bias.copy_(self.real_bias())
        return c

class MVGate(nn.Module):
    """x · σ(⟨x⟩₀): scales each multivector by a gate on its scalar part (keeps blade structure)."""
    def forward(self, x):
        B, C, H, W = x.shape; m = x.view(B, C // 4, 4, H, W)
        return (m * torch.sigmoid(m[:, :, :1])).view(B, C, H, W)

class Lift(nn.Module):
    """gray image → multivector field [I, ∂x I, ∂y I, 0]: the gradient is literally a vector."""
    def __init__(self):
        super().__init__()
        k = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]]) / 8
        self.register_buffer("kx", k[None, None]); self.register_buffer("ky", k.t()[None, None])
    def forward(self, x):
        return torch.cat([x, F.conv2d(x, self.kx, padding=1), F.conv2d(x, self.ky, padding=1), torch.zeros_like(x)], 1)

class Readout(nn.Module):
    """invariant-style readout: per multivector channel [⟨x⟩₀, |⟨x⟩₁|, ⟨x⟩₂] then linear."""
    def __init__(self, c, n_cls):
        super().__init__(); self.fc = nn.Linear(3 * c, n_cls)
    def forward(self, x):
        B, C = x.shape[:2]; m = x.mean((2, 3)).view(B, C // 4, 4)
        return self.fc(torch.cat([m[..., 0], torch.sqrt(m[..., 1] ** 2 + m[..., 2] ** 2 + 1e-6), m[..., 3]], 1))

class Standardize(nn.Module):
    """per-crop zero-mean / unit-variance: removes CCTV gain & exposure differences (inside the ONNX graph)."""
    def forward(self, x):
        m = x.mean((1, 2, 3), keepdim=True); return (x - m) / (x.std((1, 2, 3), keepdim=True) + 1e-3)

def make_clifford(width=(6, 12, 16)):
    a, b, c = width
    return nn.Sequential(Standardize(), Lift(),
                         CliffordConv2d(1, a), MVGate(), nn.AvgPool2d(2),
                         CliffordConv2d(a, b), MVGate(), nn.AvgPool2d(2),
                         CliffordConv2d(b, c), MVGate(), nn.AvgPool2d(2),
                         CliffordConv2d(c, c), MVGate(), Readout(c, 2))

def make_cnn(width=(12, 24, 32, 32)):
    """plain-CNN baseline with about the same number of trainable conv weights as make_clifford()."""
    a, b, c, d = width
    return nn.Sequential(Standardize(),
                         nn.Conv2d(1, a, 3, padding=1), nn.ReLU(), nn.AvgPool2d(2),
                         nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.AvgPool2d(2),
                         nn.Conv2d(b, c, 3, padding=1), nn.ReLU(), nn.AvgPool2d(2),
                         nn.Conv2d(c, d, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(d, 2))

def conv_params(net):
    return int(sum(m.weight.numel() for m in net.modules() if isinstance(m, (CliffordConv2d, nn.Conv2d))))

def fold_for_export(net):
    return nn.Sequential(*[m.folded() if isinstance(m, CliffordConv2d) else m for m in net])

# ─────────────────────────────── §4 SAM training ─────────────────────────────────
class SAM(torch.optim.Optimizer):
    """Sharpness-Aware Minimization: ascend to w + ρ g/‖g‖ (the linearized worst case on the ρ-ball —
    Lean: Train.sam_eps_maximizes, sam_eps_norm), then descend with the gradient taken there."""
    def __init__(self, params, base_optimizer, rho=0.05, **kw):
        super().__init__(params, dict(rho=rho, **kw))
        self.base_optimizer = base_optimizer(self.param_groups, **kw); self.param_groups = self.base_optimizer.param_groups
    @torch.no_grad()
    def first_step(self):
        gn = torch.norm(torch.stack([p.grad.norm(2) for g in self.param_groups for p in g["params"] if p.grad is not None]), 2)
        for g in self.param_groups:
            scale = g["rho"] / (gn + 1e-12)
            for p in g["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone(); p.add_(p.grad * scale)
    @torch.no_grad()
    def second_step(self):
        for g in self.param_groups:
            for p in g["params"]:
                if p.grad is not None: p.data = self.state[p]["old_p"]
        self.base_optimizer.step()

def predict(net, Xs):
    net.eval()
    with torch.no_grad(): return torch.cat([net(Xs[k:k + 512]) for k in range(0, len(Xs), 512)])

def train(make, name):
    torch.manual_seed(CFG.seed)
    net = make(); Xtr, ytr = torch.tensor(X[TR]), torch.tensor(Y[TR]); Xva = torch.tensor(X[VA])
    opt = SAM(net.parameters(), torch.optim.SGD, rho=CFG.sam_rho, lr=CFG.lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt.base_optimizer, CFG.epochs)
    lossf = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
    best, hist, t0 = (-2.0, None, -1), [], time.time()
    g = torch.Generator().manual_seed(CFG.seed)
    for ep in range(CFG.epochs):
        net.train(); perm = torch.randperm(len(Xtr), generator=g)
        for k in range(0, len(Xtr), CFG.batch):
            b = perm[k:k + CFG.batch]; xb, yb = Xtr[b], ytr[b]
            if torch.rand(1, generator=g).item() < .5: xb = torch.flip(xb, dims=[3])
            lossf(net(xb), yb).backward(); opt.first_step(); opt.zero_grad()
            lossf(net(xb), yb).backward(); opt.second_step(); opt.zero_grad()
        sched.step()
        mv = matthews_corrcoef(Y[VA], predict(net, Xva).argmax(1).numpy()); hist.append(round(float(mv), 4))
        if mv > best[0]: best = (float(mv), {k: v.clone() for k, v in net.state_dict().items()}, ep)
    net.load_state_dict(best[1]); net.eval()
    print(f"[§4] {name}: SAM ρ={CFG.sam_rho}, {CFG.epochs} epochs in {time.time() - t0:.0f} s · val MCC per epoch {hist} → epoch {best[2]}")
    return net, {"val_mcc": best[0], "best_epoch": best[2], "val_mcc_history": hist, "train_s": round(time.time() - t0, 1)}

# ─────────────────────────────── §5 evaluation: MCC, CI, sharpness ───────────────
def mcc_ci(y, p, n, seed):
    r = np.random.default_rng(seed); s = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); s.append(matthews_corrcoef(y[i], p[i]))
    return [float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))]

def sharpness(net):
    """test-loss increase at the SAM point w + ρ g/‖g‖ (first-order worst case on the ρ-ball)."""
    Xt, yt = torch.tensor(X[TE]), torch.tensor(Y[TE]); lossf = nn.CrossEntropyLoss()
    net.eval(); net.zero_grad(); l0 = lossf(net(Xt), yt); l0.backward()
    ps = [p for p in net.parameters() if p.grad is not None]; gn = torch.norm(torch.stack([p.grad.norm() for p in ps]))
    with torch.no_grad():
        old = [p.clone() for p in ps]
        for p in ps: p.add_(p.grad * CFG.sam_rho / (gn + 1e-12))
        l1 = lossf(net(Xt), yt)
        for p, o in zip(ps, old): p.copy_(o)
    net.zero_grad()
    return {"rho": CFG.sam_rho, "test_loss": l0.item(), "test_loss_at_sam_point": l1.item(), "increase": (l1 - l0).item()}

def evaluate(net, info):
    pred = predict(net, torch.tensor(X[TE])).argmax(1).numpy(); y = Y[TE]
    cm = confusion_matrix(y, pred, labels=[0, 1])
    return {**info, "test_mcc": float(matthews_corrcoef(y, pred)), "test_mcc_ci95": mcc_ci(y, pred, CFG.bootstrap, CFG.seed),
            "confusion[[tn,fp],[fn,tp]]": cm.tolist(), "conv_params": conv_params(net), "sharpness": sharpness(net)}

NET, info = train(make_clifford, "Clifford-conv Cl(2,0)")
REPORT["clifford"] = evaluate(NET, info)
if CFG.baseline:
    CNN, info = train(make_cnn, "plain CNN baseline")
    REPORT["baseline_cnn"] = evaluate(CNN, info)
for k in ("clifford", "baseline_cnn"):
    if k in REPORT:
        r = REPORT[k]
        print(f"[§5] {k:13s} test MCC {r['test_mcc']:.3f}  95% CI [{r['test_mcc_ci95'][0]:.3f}, {r['test_mcc_ci95'][1]:.3f}]  "
              f"conv params {r['conv_params']}  sharpness +{r['sharpness']['increase']:.4f}")

# ─────────────────────────────── §6 export: fold → ONNX, parity, card, golden ────
folded = fold_for_export(NET).eval()
Xte = torch.tensor(X[TE])
with torch.no_grad(): d_fold = float((folded(Xte) - NET(Xte)).abs().max())
assert d_fold < 1e-4, d_fold                                              # Train.fold_correct, numerically
onnx_path = OUT / "models/ppe_clifford_folded.onnx"
kw = dict(input_names=["crop"], output_names=["logits"], dynamic_axes={"crop": {0: "n"}, "logits": {0: "n"}}, opset_version=17)
try: torch.onnx.export(folded, Xte[:1], str(onnx_path), dynamo=False, **kw)
except TypeError: torch.onnx.export(folded, Xte[:1], str(onnx_path), **kw)
onnx.checker.check_model(onnx.load(str(onnx_path)))
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
lg_ort = np.concatenate([sess.run(None, {"crop": X[TE][k:k + 256]})[0] for k in range(0, len(TE), 256)])
with torch.no_grad(): d_ort = float(np.abs(lg_ort - NET(Xte).numpy()).max())
assert d_ort < 1e-3, d_ort
assert matthews_corrcoef(Y[TE], lg_ort.argmax(1)) == REPORT["clifford"]["test_mcc"], "ONNX changes test predictions"
model_sha = hashlib.sha256(onnx_path.read_bytes()).hexdigest()
t0 = time.perf_counter()
for _ in range(50): sess.run(None, {"crop": X[TE][:8]})
ms8 = (time.perf_counter() - t0) / 50 * 1000

# golden vectors for crates/vg360-infer/tests/golden.rs: test crops quantized to u8 (exactly reproducible)
G = TE[np.linspace(0, len(TE) - 1, 16).astype(int)]
g_u8 = np.round(X[G, 0] * 255).astype(np.uint8)
g_lg = sess.run(None, {"crop": (g_u8.astype(np.float32) / 255)[:, None]})[0]
g_p = np.exp(g_lg - g_lg.max(1, keepdims=True)); g_p = g_p[:, 1] / g_p.sum(1)
golden = {"model_sha256": model_sha, "size": CFG.size, "input": "u8 crop / 255, shape [1,1,48,48]",
          "crops": [{"label": int(Y[i]), "u8": c.flatten().tolist(), "logits": l.tolist(), "p_helmet": float(p)}
                    for i, c, l, p in zip(G, g_u8, g_lg, g_p)]}
(OUT / "testdata/golden/ppe.json").write_text(json.dumps(golden))

floor = FLOORS[CFG.severity]; c = REPORT["clifford"]
card = {
    "model": "ppe_clifford_folded.onnx", "task": "helmet / no-helmet on 48×48 gray head crops (input: crop, output: logits [no_helmet, helmet])",
    "sha256": model_sha, "created_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "architecture": "Standardize → Lift [I, ∂xI, ∂yI, 0] → 4× CliffordConv2d Cl(2,0) (folded to Conv2d) + MVGate → invariant readout",
    "training": {"optimizer": "SAM(SGD momentum 0.9, wd 5e-4)", "rho": CFG.sam_rho, "lr": CFG.lr, "epochs": CFG.epochs,
                 "batch": CFG.batch, "label_smoothing": CFG.label_smoothing, "seed": CFG.seed, "selection": "best val MCC",
                 "torch": torch.__version__, "onnx_opset": 17},
    "data": REPORT["data"],
    "metrics": {k: c[k] for k in ("test_mcc", "test_mcc_ci95", "confusion[[tn,fp],[fn,tp]]", "val_mcc", "best_epoch", "sharpness")},
    "baseline_cnn": {k: REPORT["baseline_cnn"][k] for k in ("test_mcc", "test_mcc_ci95", "conv_params", "sharpness")} if CFG.baseline else None,
    "conv_params": {"clifford_trainable": c["conv_params"], "folded_real": conv_params(folded)},
    "parity": {"max_abs_torch_vs_folded": d_fold, "max_abs_torch_vs_onnxruntime": d_ort, "onnx_ms_per_8_crops": round(ms8, 3)},
    "gate": {"rule": "PPE", "severity": CFG.severity, "mcc_floor": floor, "meets_floor": c["test_mcc"] >= floor,
             "note": "the rule is armed only when also approved in iogp-editor; synthetic data never arms it" if SOURCE.startswith("synthetic") else ""},
    "lean": ["VG360.Clifford.gp_assoc", "VG360.Clifford.idxTable_eq", "VG360.Clifford.sgnTable_eq", "VG360.Train.fold_correct",
             "VG360.Train.fold_conv", "VG360.Train.mcc_abs_le_one", "VG360.Train.mcc_swap", "VG360.Train.sam_eps_maximizes"],
    "golden": "testdata/golden/ppe.json",
}
(OUT / "models/ppe_clifford_folded.card.json").write_text(json.dumps(card, indent=1))
REPORT["export"] = {"sha256": model_sha, **card["parity"], "out": str(OUT)}
print(f"[§6] folded ≡ trained (max |Δ| {d_fold:.1e}), ONNX ≡ torch (max |Δ| {d_ort:.1e}), {ms8:.2f} ms / 8 crops, "
      f"sha256 {model_sha[:16]}… → {OUT / 'models'}")

# the detector golden file carries the PPE probabilities too: refresh it with the new model when writing the repo
if OUT == ROOT and (ROOT / "models/yolox_nano.onnx").exists():
    r = subprocess.run([sys.executable, str(ROOT / "scripts/export_golden.py")], capture_output=True, text=True, cwd=ROOT)
    print("[§6] testdata/golden/infer.json:", r.stdout.strip() or r.stderr[-500:])

# ─────────────────────────────── §7 promotion gate ───────────────────────────────
print(f"[§7] PPE ({CFG.severity}, floor {floor}): test MCC {c['test_mcc']:.3f} → "
      + ("meets the floor" if card["gate"]["meets_floor"] else "BELOW the floor — the rule stays in shadow mode")
      + ("; trained on synthetic crops: retrain on real site crops (CFG.ppe_dir / CFG.voc_dir) before arming" if SOURCE.startswith("synthetic") else ""))

# ─────────────────────────────── §8 ask the console (OpenAI-compatible) ──────────
if CFG.console_url:
    import urllib.request
    body = json.dumps({"model": "vg360-ops", "messages": [{"role": "user", "content": "rules mcc"}]}).encode()
    req = urllib.request.Request(CFG.console_url.rstrip("/") + "/v1/chat/completions", body, {"content-type": "application/json"})
    print("[§8] console says:\n" + json.load(urllib.request.urlopen(req, timeout=10))["choices"][0]["message"]["content"])
else:
    print("[§8] set VG360_URL to ask a running console via /v1/chat/completions")

# ─────────────────────────────── §9 site footage: detector + PPE audit, crops to label ──
# Real footage is where the synthetic smoke test stops being informative. For each sampled frame: the
# runtime's detector (YOLOX-Nano, gray, full frame + 2×2 tiles, same pre/post-processing as vg360-infer)
# at a low threshold, the new PPE model on every head crop, and a contact sheet. Person crops and frames go
# to footage_audit/ for labelling (helmet / no_helmet folders → CFG.ppe_dir; frames → box labels for a
# detector fine-tune). Site imagery shows identifiable workers: keep footage_audit/ out of git.
def letterbox(img, size=416):
    h, w = img.shape[:2]; r = min(size / h, size / w)
    out = np.full((size, size, 3), 114, np.uint8); out[:int(h * r), :int(w * r)] = cv2.resize(img, (int(w * r), int(h * r)))
    return out, r

class YOLOX:
    def __init__(self, path, size=416):
        self.sess, self.size = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"]), size
        g, s = [], []
        for st in (8, 16, 32):
            n = size // st; xv, yv = np.meshgrid(np.arange(n), np.arange(n))
            g.append(np.stack((xv, yv), 2).reshape(-1, 2)); s.append(np.full((n * n, 1), st))
        self.grids, self.strides = np.concatenate(g), np.concatenate(s)
    def __call__(self, gray, conf):
        lb, r = letterbox(cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR), self.size)
        o = self.sess.run(None, {"images": lb.transpose(2, 0, 1)[None].astype(np.float32)})[0][0]
        xy = (o[:, :2] + self.grids) * self.strides; wh = np.exp(o[:, 2:4]) * self.strides
        sc = o[:, 4:5] * o[:, 5:]; keep = (sc.argmax(1) == 0) & (sc.max(1) > conf)
        b = np.concatenate([xy - wh / 2, xy + wh / 2], 1)[keep] / r
        return b, sc.max(1)[keep]

def nms(b, s, conf, thr=0.45):
    if not len(b): return b, s
    i = np.array(cv2.dnn.NMSBoxes(np.c_[b[:, :2], b[:, 2:] - b[:, :2]].tolist(), s.tolist(), conf, thr)).reshape(-1)
    return b[i], s[i]

def detect_tiled(det, g, conf, tiles=2, overlap=0.25):
    B, S = [list(x) for x in det(g, conf)]
    H, W = g.shape; th, tw = int(H / tiles * (1 + overlap)), int(W / tiles * (1 + overlap))
    for i in range(tiles):
        for j in range(tiles):
            y0, x0 = min(int(i * H / tiles), H - th), min(int(j * W / tiles), W - tw)
            b, c = det(np.ascontiguousarray(g[y0:y0 + th, x0:x0 + tw]), conf)
            B += [bb + np.array([x0, y0, x0, y0]) for bb in b]; S += list(c)
    return nms(np.array(B).reshape(-1, 4), np.array(S), conf)

def head_crop(g, box, size):
    x1, y1, x2, y2 = box; h = y2 - y1; s = max(0.24 * h, 8); cx = (x1 + x2) / 2; top = y1 - 0.03 * h
    xa, ya = int(max(cx - s / 2, 0)), int(max(top, 0)); xb, yb = int(min(cx + s / 2, g.shape[1])), int(min(top + s, g.shape[0]))
    if xb - xa < 4 or yb - ya < 4: return None
    return cv2.resize(g[ya:yb, xa:xb], (size, size)).astype(np.float32) / 255

def audit_footage(paths):
    det_path = (ROOT / "models/yolox_nano.onnx") if ROOT is not None and (ROOT / "models/yolox_nano.onnx").exists() else OUT / "models/yolox_nano.onnx"
    if not det_path.exists():
        import urllib.request; urllib.request.urlretrieve(CFG.detector_url, det_path)
    det, aud = YOLOX(det_path), OUT / "footage_audit"
    (aud / "frames").mkdir(parents=True, exist_ok=True); (aud / "crops_unlabelled").mkdir(exist_ok=True)
    rows, sheet = [], []
    for vi, path in enumerate(paths):
        vf = f"fps={CFG.footage_fps}" + (f",crop={CFG.footage_crop}" if CFG.footage_crop else "")
        subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", path, "-an", "-vf", vf, str(aud / f"frames/v{vi}_%04d.png")], check=True)
        for fp in sorted((aud / "frames").glob(f"v{vi}_*.png")):
            im = cv2.imread(str(fp)); g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
            if g.mean() < 8: continue                                    # black transition frames
            b, s = detect_tiled(det, g, conf=0.10)
            for k, (bb, sc) in enumerate(zip(b, s)):
                c = head_crop(g, bb, CFG.size)
                p = None
                if c is not None:
                    lg = sess.run(None, {"crop": c[None, None]})[0][0]; e = np.exp(lg - lg.max()); p = float(e[1] / e.sum())
                x1, y1, x2, y2 = [int(v) for v in bb]
                crop = im[max(y1, 0):max(y2, 1), max(x1, 0):max(x2, 1)]
                name = f"{fp.stem}_{k}_s{sc:.2f}.png"
                if crop.size: cv2.imwrite(str(aud / "crops_unlabelled" / name), crop)
                rows.append({"frame": fp.name, "box": [x1, y1, x2, y2], "score": float(sc), "p_helmet": p, "crop": name})
                if crop.size and len(sheet) < 96:
                    t = cv2.resize(crop, (120, 160)); cv2.putText(t, f"{sc:.2f} h{p if p is not None else -1:.2f}", (2, 14), 0, .42, (0, 255, 255), 1)
                    sheet.append(t)
    scores = np.array([r["score"] for r in rows])
    rep = {"frames": len(set(r["frame"] for r in rows)), "detections_at_0.10": len(rows),
           "at_runtime_threshold_0.35": int((scores >= 0.35).sum()) if len(scores) else 0,
           "score_quantiles": np.quantile(scores, [.5, .9, .99]).round(3).tolist() if len(scores) else [],
           "out": str(aud), "next": "move person crops into helmet/ and no_helmet/ (→ CFG.ppe_dir); label person boxes on frames/ "
                                   "(YOLO format) to fine-tune the detector with SAM"}
    if sheet:
        while len(sheet) % 12: sheet.append(np.zeros_like(sheet[0]))
        cv2.imwrite(str(aud / "contact_sheet.jpg"), np.vstack([np.hstack(sheet[i:i + 12]) for i in range(0, len(sheet), 12)]))
    (aud / "detections.json").write_text(json.dumps(rows))
    return rep

if CFG.footage:
    REPORT["footage"] = audit_footage([p.strip() for p in CFG.footage.split(",") if p.strip()])
    print(f"[§9] footage audit: {json.dumps(REPORT['footage'])}")
else:
    print("[§9] set VG360_FOOTAGE (and CFG.footage_crop for screen recordings) to audit site footage")
(OUT / "testdata/golden/train_report.json").write_text(json.dumps(REPORT, indent=1, default=str))